# Урок 5 — AsyncIO и Многопроцессорность
### Ноутбук с примерами кода к лекции

Этот ноутбук содержит рабочие примеры кода, соответствующие разделам конспекта `urok5_asyncio_multiprocessing.md`.
Все примеры можно запускать по ходу лекции для демонстрации студентам.

**Важно:** в Jupyter уже запущен свой event loop, поэтому конструкции с `await` можно писать
прямо в ячейке верхнего уровня (без `asyncio.run()`). Там, где нужен `asyncio.run()` для
самостоятельного запуска вне Jupyter, это отмечено отдельно.


## 2. Конкурентность vs Параллелизм, и GIL

Демонстрация: почему потоки НЕ ускоряют CPU-ограниченную задачу в Python из-за GIL.


In [ ]:
import time
import threading

def cpu_bound(n: int) -> int:
    return sum(i * i for i in range(n))

N = 10_000_000

# Последовательно, без потоков
start = time.time()
cpu_bound(N)
cpu_bound(N)
print(f"Последовательно (2x): {time.time() - start:.2f} сек")

# С потоками — ожидаем ускорения, но его почти не будет из-за GIL
start = time.time()
t1 = threading.Thread(target=cpu_bound, args=(N,))
t2 = threading.Thread(target=cpu_bound, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"С потоками (2x):     {time.time() - start:.2f} сек  <- почти не быстрее!")


**Вывод для обсуждения:** несмотря на два потока, время практически не уменьшилось —
GIL позволяет исполнять байткод только одному потоку одновременно. Дальше в разделе 6
покажем, как `multiprocessing` решает именно эту проблему.

## 3. Threading — краткий обзор для контраста

Потоки *хорошо* работают для I/O-ограниченных задач, потому что блокирующие вызовы
(например, `time.sleep`, сетевые запросы) освобождают GIL.


In [ ]:
import threading
import time

def download(name: str, delay: float) -> None:
    print(f"Начало загрузки {name}")
    time.sleep(delay)  # имитация сетевого запроса
    print(f"Загрузка {name} завершена")

threads = [
    threading.Thread(target=download, args=(f"файл_{i}", 2))
    for i in range(5)
]

start = time.time()
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"Всего: {time.time() - start:.1f} сек")  # ~2 сек, а не 10


## 4. AsyncIO: концепция и синтаксис

### 4.1–4.2 Базовый синтаксис

В Jupyter event loop уже запущен, поэтому просто используем `await` в ячейке напрямую.


In [ ]:
import asyncio
import time

async def download(name: str, delay: float) -> None:
    print(f"Начало загрузки {name}")
    await asyncio.sleep(delay)  # НЕ time.sleep()! — не блокирует событийный цикл
    print(f"Загрузка {name} завершена")

async def main() -> None:
    start = time.time()
    await asyncio.gather(
        download("файл_1", 2),
        download("файл_2", 2),
        download("файл_3", 2),
    )
    print(f"Всего: {time.time() - start:.1f} сек")  # ~2 сек, а не 6

await main()  # в обычном .py скрипте здесь было бы: asyncio.run(main())


**Частая ошибка новичков — обязательно показать на доске:**

```python
# НЕПРАВИЛЬНО — time.sleep() блокирует весь событийный цикл
async def bad(delay):
    time.sleep(delay)   # ❌ Блокирует ВСЁ, а не только эту задачу

# ПРАВИЛЬНО
async def good(delay):
    await asyncio.sleep(delay)   # ✅ Освобождает цикл событий другим задачам
```

Демонстрация ниже показывает разницу на практике: если запустить несколько "плохих"
корутин параллельно, они всё равно выполнятся последовательно.


In [ ]:
import asyncio
import time

async def bad_task(name, delay):
    print(f"[bad] старт {name}")
    time.sleep(delay)  # блокирует весь event loop!
    print(f"[bad] финиш {name}")

async def good_task(name, delay):
    print(f"[good] старт {name}")
    await asyncio.sleep(delay)  # не блокирует event loop
    print(f"[good] финиш {name}")

print("--- Демонстрация BAD (time.sleep внутри async) ---")
start = time.time()
await asyncio.gather(bad_task("A", 1), bad_task("B", 1))
print(f"Время: {time.time() - start:.2f} сек (ожидаемо ~2 сек — задачи НЕ пересеклись)\n")

print("--- Демонстрация GOOD (asyncio.sleep) ---")
start = time.time()
await asyncio.gather(good_task("A", 1), good_task("B", 1))
print(f"Время: {time.time() - start:.2f} сек (ожидаемо ~1 сек — задачи пересеклись)")


### 4.3 Реальный пример: пакетные API-запросы

Требует установки `aiohttp` (`pip install aiohttp`). Пример обращается к публичному
тестовому API `httpbin.org`, который с задержкой возвращает JSON — удобно для демонстрации
конкурентных запросов без необходимости в реальном платном API.


In [ ]:
# Если aiohttp не установлен, раскомментируйте строку ниже:
# %pip install aiohttp -q

import aiohttp
import asyncio
import time

async def fetch_delayed(session: aiohttp.ClientSession, delay: float) -> dict:
    # httpbin.org/delay/N возвращает ответ через N секунд — удобно для демо
    url = f"https://httpbin.org/delay/{delay}"
    async with session.get(url) as response:
        return await response.json()

async def fetch_all(n_requests: int, delay: float) -> list:
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_delayed(session, delay) for _ in range(n_requests)]
        return await asyncio.gather(*tasks)

start = time.time()
results = await fetch_all(n_requests=5, delay=1)
print(f"5 запросов с задержкой 1 сек каждый, выполнено за {time.time() - start:.1f} сек")
print(f"(последовательно заняло бы ~5 сек)")


### 4.4 `asyncio.gather` vs `asyncio.as_completed`

In [ ]:
import asyncio
import random

async def fetch_price(ticker: str) -> dict:
    delay = random.uniform(0.5, 2.0)
    await asyncio.sleep(delay)
    return {"ticker": ticker, "delay": round(delay, 2)}

tickers = ["AAPL", "MSFT", "GOOG", "AMZN"]

print("--- gather: ждём все сразу, порядок сохранён ---")
results = await asyncio.gather(*(fetch_price(t) for t in tickers))
for r in results:
    print(r)

print("\n--- as_completed: обрабатываем по мере готовности ---")
for coro in asyncio.as_completed([fetch_price(t) for t in tickers]):
    result = await coro
    print(f"Готово: {result}")


### 4.5 Ограничение конкурентности: `asyncio.Semaphore`

Полезно, когда API разрешает не более N одновременных запросов.


In [ ]:
import aiohttp
import asyncio
import time

async def fetch_with_limit(session, delay, semaphore):
    async with semaphore:
        url = f"https://httpbin.org/delay/{delay}"
        async with session.get(url) as response:
            return await response.json()

async def fetch_all_limited(n_requests: int, delay: float, max_concurrent: int):
    semaphore = asyncio.Semaphore(max_concurrent)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_with_limit(session, delay, semaphore) for _ in range(n_requests)]
        return await asyncio.gather(*tasks)

start = time.time()
# 10 запросов, но не более 3 одновременно
await fetch_all_limited(n_requests=10, delay=1, max_concurrent=3)
print(f"10 запросов, макс. 3 одновременно: {time.time() - start:.1f} сек")
print("(ожидаемо ~4 сек: 3+3+3+1 волны по ~1 сек)")


## 5. Асинхронные контекстные менеджеры и генераторы

### 5.1 Асинхронные контекстные менеджеры — для ресурсов, требующих завершения


In [ ]:
import asyncio

class AsyncDatabaseConnection:
    async def __aenter__(self):
        print("Открываем соединение...")
        await asyncio.sleep(0.1)  # имитация асинхронного подключения
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("Закрываем соединение...")
        await asyncio.sleep(0.1)  # имитация асинхронного закрытия

    async def query(self, sql: str) -> list:
        await asyncio.sleep(0.2)
        return [{"result": "данные"}]

async def main():
    async with AsyncDatabaseConnection() as conn:
        data = await conn.query("SELECT * FROM prices")
        print(data)
    # соединение автоматически и асинхронно закрыто

await main()


### 5.2 Асинхронные генераторы — для потоковых данных

In [ ]:
import asyncio

async def stream_prices(ticker: str):
    for i in range(5):
        await asyncio.sleep(0.5)  # имитация поступления новой цены
        yield {"ticker": ticker, "price": 100 + i}

async def main():
    async for price in stream_prices("AAPL"):
        print(price)

await main()


## 6. Многопроцессорность

⚠️ **Важно про Jupyter:** `multiprocessing` на Windows/macOS требует, чтобы функции-воркеры
были определены в отдельном `.py` файле и импортированы (из-за способа старта процессов
`spawn`), иначе может возникнуть ошибка pickle при определении функции прямо в ячейке ноутбука.
Ниже это сделано через отдельный вспомогательный модуль `mp_worker.py`, который создаётся
первой ячейкой этого раздела — это и есть стандартный обходной путь для демонстрации
multiprocessing внутри Jupyter.


In [ ]:
%%writefile mp_worker.py
import pandas as pd

def worker(n: int) -> int:
    return sum(i**2 for i in range(n))

def process_file(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)
    df["normalized"] = (df["value"] - df["value"].mean()) / df["value"].std()
    return df


### 6.1 Зачем — обход GIL для CPU-задач

In [ ]:
from multiprocessing import Process
from mp_worker import worker

if __name__ == "__main__":
    p = Process(target=worker, args=(1_000_000,))
    p.start()
    p.join()   # ждать завершения
    print("Процесс завершён")


**Зачем нужен `if __name__ == "__main__"`?**
В Windows новые процессы создаются путём импорта модуля — без этой защиты возникает
бесконечная рекурсия, так как каждый дочерний процесс порождает новые дочерние.

### 6.2 Практичный инструмент: `multiprocessing.Pool`

Демонстрация с реальным ускорением CPU-ограниченной задачи (несколько вызовов `worker`).


In [ ]:
import time
from multiprocessing import Pool
from mp_worker import worker

N = 5_000_000
inputs = [N] * 4

if __name__ == "__main__":
    # Последовательно
    start = time.time()
    seq_results = [worker(n) for n in inputs]
    seq_time = time.time() - start
    print(f"Последовательно: {seq_time:.2f} сек")

    # Параллельно через Pool
    start = time.time()
    with Pool(processes=4) as pool:
        par_results = pool.map(worker, inputs)
    par_time = time.time() - start
    print(f"С Pool(4):       {par_time:.2f} сек")
    print(f"Ускорение:       {seq_time / par_time:.1f}x")


Пример с DataFrame (как в конспекте) — требует наличия CSV-файлов на диске.
Здесь для демонстрации сначала создадим тестовые файлы.


In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs("demo_data", exist_ok=True)
file_paths = []
for i in range(5):
    df = pd.DataFrame({"value": np.random.randn(1000)})
    path = f"demo_data/data_{i}.csv"
    df.to_csv(path, index=False)
    file_paths.append(path)

print(f"Создано {len(file_paths)} тестовых CSV-файлов в ./demo_data/")


In [ ]:
from multiprocessing import Pool
from mp_worker import process_file
import pandas as pd

if __name__ == "__main__":
    with Pool(processes=4) as pool:
        results = pool.map(process_file, file_paths)

    final_df = pd.concat(results, ignore_index=True)
    print(final_df.head())
    print(f"\nВсего строк после объединения: {len(final_df)}")


### 6.4 `ProcessPoolExecutor` — альтернативный, более современный API

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from mp_worker import process_file

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(process_file, file_paths))
    print(f"Обработано файлов: {len(results)}")


## 7. Дерево решений — памятка

```
Задача ограничена I/O? (сеть, диск, API-запросы)
├── Да → asyncio (или threading для блокирующих библиотек)
└── Нет → Ограничена CPU? (вычисления, парсинг, ML)
           ├── Да → multiprocessing.Pool
           │         ├── Но: данные огромны для сериализации? → рассмотрите векторизацию (NumPy)
           │         └── Или: каждая задача < 100 мс? → накладные расходы не оправданы
           └── Нет → последовательный код вполне подойдёт
```

| Задача | Правильный инструмент | Почему |
|---|---|---|
| Скачать 100 CSV с S3 | asyncio / aiohttp | Ограничена I/O |
| Пакетные API-запросы к Bloomberg | asyncio | Ограничена I/O |
| Параллельная предобработка 50 CSV-файлов | multiprocessing.Pool | Ограничена CPU |
| Обучение ML-моделей с разными гиперпараметрами | multiprocessing.Pool | Ограничена CPU |
| Нормализация DataFrame из 10M строк | Векторизация NumPy | Полностью избегаем накладных расходов |
| Потоковая обработка событий в реальном времени | asyncio | Конкурентный I/O |


## 8. Практика

Раскомментируйте и допишите решения ниже.


In [ ]:
# Упражнение 1:
# Перепишите последовательный цикл `for ticker in tickers: requests.get(...)`
# на asyncio + aiohttp, добавив Semaphore(5).

# ваш код здесь


In [ ]:
# Упражнение 2:
# Возьмите функцию, применяющую .apply() к DataFrame построчно,
# и перепишите её через multiprocessing.Pool, замерив время до и после.

# ваш код здесь


**Упражнение 3 (устно/письменно):**
Почему нельзя просто добавить больше потоков, чтобы ускорить CPU-ограниченную задачу в Python?

_(Ответ: из-за GIL — только один поток исполняет байткод Python в любой момент времени,
поэтому потоки не дают параллелизма для вычислений, только для ожидания I/O.)_


## Итоговые тезисы урока

- GIL — причина, почему в Python потоки не помогают с вычислениями, но помогают с ожиданием.
- asyncio — для I/O, один поток, кооперативное переключение через `await`.
- multiprocessing — для CPU, реальный параллелизм, но с расходами на сериализацию.
- Всегда сначала проверяйте, нельзя ли обойтись векторизацией (NumPy/Pandas) — она часто
  быстрее и проще, чем любая многопроцессорность.


In [ ]:
# Очистка тестовых файлов (по желанию)
import shutil
shutil.rmtree("demo_data", ignore_errors=True)
import os
if os.path.exists("mp_worker.py"):
    os.remove("mp_worker.py")
print("Тестовые файлы удалены")
